# **Pixel-based baselines**

#Setting the environment
1. Mounting Google Drive
2. Cloning from GitHub
3. Installing requirements

In [ ]:
# Setup centralizzato: mount Drive + clone/update repo + install deps via colab_setup.py.
# (vedi sezione "Setup su Colab" del README). Idempotente sullo stesso server Colab.
from google.colab import drive
drive.mount("/content/drive")

import os, sys, subprocess
REPO = "/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject"
BRANCH = "finetuning/coco-to-cityscapes"

# Prima volta: clona la repo, cosi' colab_setup diventa importabile.
if not os.path.exists(REPO + "/.git"):
    subprocess.run(["git", "clone", "--branch", BRANCH,
        "https://github.com/ChiaraApolito/MaskArchitectureAnomaly_CourseProject.git", REPO])

sys.path.insert(0, REPO + "/notebook")   # colab_setup.py vive in notebook/
from colab_setup import bootstrap
P = bootstrap(branch=BRANCH, task="task7")   # mount + update repo + install deps (+ ood_metrics)


In [ ]:
from pathlib import Path
import zipfile

# Path derivati da P (ritornato da bootstrap).
PROJECT_ROOT = P.project
ANOMALY_ZIP = P.anomaly_zip
LOCAL_DATA_DIR = Path("/content/anomaly_data")

print("Zip exists:", ANOMALY_ZIP.exists())
if not LOCAL_DATA_DIR.exists():
    print("Extracting zip temporarily to /content...")
    with zipfile.ZipFile(ANOMALY_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)
    print("Done.")
else:
    print("Already extracted in this runtime.")

DATA_ROOT = LOCAL_DATA_DIR / "Validation_Dataset"
print("DATA_ROOT exists:", DATA_ROOT.exists())

TRAINED_MODELS_DIR = P.project / "trained_models"
ERFNET_WEIGHTS = TRAINED_MODELS_DIR / "erfnet_pretrained.pth"
EVAL_DIR = P.eval

In [4]:
print("Project exists:", PROJECT_ROOT.exists())
print("Eval exists:", EVAL_DIR.exists())
print("Dataset exists:", DATA_ROOT.exists())
print("Trained models exists:", TRAINED_MODELS_DIR.exists())
print("ERFNet weights exists:", ERFNET_WEIGHTS.exists())

Project exists: True
Eval exists: True
Dataset exists: True
Trained models exists: True
ERFNet weights exists: True


In [5]:
datasets = {
    "FS_LostFound_full": DATA_ROOT / "FS_LostFound_full" / "images" / "*.png",
    "fs_static": DATA_ROOT / "fs_static" / "images" / "*.jpg",
    "RoadAnomaly": DATA_ROOT / "RoadAnomaly" / "images" / "*.jpg",
    "RoadAnomaly21": DATA_ROOT / "RoadAnomaly21" / "images" / "*.png",
    "RoadObsticle21": DATA_ROOT / "RoadObsticle21" / "images" / "*.webp",
}

In [6]:
import glob

for name, pattern in datasets.items():
    files = glob.glob(str(pattern))
    print(name, len(files), "images")

FS_LostFound_full 100 images
fs_static 30 images
RoadAnomaly 60 images
RoadAnomaly21 10 images
RoadObsticle21 30 images


In [7]:
methods = ["maxlogit", "msp", "entropy"]

In [8]:
print("EVAL_DIR:", EVAL_DIR)
print("erfnet exists:", (EVAL_DIR / "erfnet.py").exists())

EVAL_DIR: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eval
erfnet exists: True


In [9]:
import subprocess

for dataset_name, input_pattern in datasets.items():
    for method in methods:
        print(f"\nRunning {method} on {dataset_name}")

        cmd = [
            "python", str(EVAL_DIR / "evalAnomaly.py"),
            "--input", str(input_pattern),
            "--loadDir", str(TRAINED_MODELS_DIR) + "/",
            "--loadWeights", "erfnet_pretrained.pth",
            "--method", method,
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)

        if result.stderr:
            print("STDERR:", result.stderr)


Running maxlogit on FS_LostFound_full
Input: /content/anomaly_data/Validation_Dataset/FS_LostFound_full/images/*.png
Loading model: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/trained_models/erfnet.py
Loading weights: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/trained_models/erfnet_pretrained.pth
Model and weights LOADED successfully
/content/anomaly_data/Validation_Dataset/FS_LostFound_full/images/39.png
Unique values AFTER mapping: [  0   1 255]
/content/anomaly_data/Validation_Dataset/FS_LostFound_full/images/24.png
Unique values AFTER mapping: [  0   1 255]
/content/anomaly_data/Validation_Dataset/FS_LostFound_full/images/93.png
Unique values AFTER mapping: [  0   1 255]
/content/anomaly_data/Validation_Dataset/FS_LostFound_full/images/1.png
Unique values AFTER mapping: [  0   1 255]
/content/anomaly_data/Validation_Dataset/FS_LostFound_full/images/86.png
Unique values AFTER mapping: [  0   1 255]
/content/anomaly_data/Validation_Dataset/FS_LostF